# Depth Model Interface — Handcrafted Feature Pipeline

This notebook turns your handcrafted audio-feature pipeline into a drop-in song interface.

Workflow:
1. Save PCA artifacts from your batch feature pipeline outputs.
2. Load the saved `depth_model`.
3. Upload one song.
4. Extract the same librosa feature blocks used in `03_MTG_feature_pipeline.ipynb`.
5. Apply saved PCA transforms.
6. Predict depth with the saved model.

**Important:** a one-song app cannot fit PCA on the uploaded song. It must reuse PCA objects saved from the same feature pipeline used to create the model-ready feature matrix.

## 1. Install dependencies

In [ ]:
# Uncomment if needed:
# %pip install -q librosa pandas numpy scikit-learn joblib gradio xgboost soundfile

## 2. Imports and project paths

Edit `FEATURE_DIR`, `MODEL_DIR`, and `MODEL_PATH` as needed.

In [1]:
from pathlib import Path
import math
import warnings

import joblib
import librosa
import numpy as np
import pandas as pd
import gradio as gr

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "features"      # raw/reduced feature CSVs from 03_MTG_feature_pipeline
MODEL_DIR = PROJECT_ROOT / "models"          # depth_model + PCA artifacts
MODEL_PATH = MODEL_DIR / "depth_model.joblib"

MODEL_DIR.mkdir(exist_ok=True)
FEATURE_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)
print("MODEL_DIR:", MODEL_DIR)

PROJECT_ROOT: /Users/Type3Strength/Triple10_Externship/Notebooks
FEATURE_DIR: /Users/Type3Strength/Triple10_Externship/Notebooks/features
MODEL_DIR: /Users/Type3Strength/Triple10_Externship/Notebooks/models


## 3. Save PCA artifacts from existing raw feature CSVs

Run this after `03_MTG_feature_pipeline.ipynb` has created the raw block files:

- `mid_level_mfccs_all.csv`
- `mid_level_chroma_all.csv`
- `mid_level_cent_all.csv`
- `mid_level_bw_all.csv`
- `mid_level_contrast_all.csv`
- `mid_level_flat_all.csv`
- `mid_level_rolloff_all.csv`
- `mid_level_tonnetz_all.csv`

This fits PCA and saves each PCA transformer into `models/`.

In [2]:
from sklearn.decomposition import PCA

RAW_BLOCK_FILES = {
    "mfcc": "mid_level_mfccs_all.csv",
    "chroma": "mid_level_chroma_all.csv",
    "cent": "mid_level_cent_all.csv",
    "bw": "mid_level_bw_all.csv",
    "contrast": "mid_level_contrast_all.csv",
    "flat": "mid_level_flat_all.csv",
    "rolloff": "mid_level_rolloff_all.csv",
    "tonnetz": "mid_level_tonnetz_all.csv",
}

PCA_FILES = {
    "mfcc": "pca_mfcc.joblib",
    "chroma": "pca_chroma.joblib",
    "cent": "pca_cent.joblib",
    "bw": "pca_bw.joblib",
    "contrast": "pca_contrast.joblib",
    "flat": "pca_flat.joblib",
    "rolloff": "pca_rolloff.joblib",
    "tonnetz": "pca_tonnetz.joblib",
}

REDUCED_OUTPUT_FILES = {
    "mfcc": "mid_levle_mfccs_reduc.csv",
    "chroma": "mid_levle_chroma_reduc.csv",
    "cent": "mid_levle_cent_reduc.csv",
    "bw": "mid_levle_bw_reduc.csv",
    "contrast": "mid_levle_contrast_reduc.csv",
    "flat": "mid_levle_flat_reduc.csv",
    "rolloff": "mid_levle_rolloff_reduc.csv",
    "tonnetz": "mid_levle_tonnetz_reduc.csv",
}

def fit_and_save_pca_artifacts(feature_dir=FEATURE_DIR, model_dir=MODEL_DIR, n_components=50):
    feature_dir = Path(feature_dir)
    model_dir = Path(model_dir)
    model_dir.mkdir(exist_ok=True)
    pcas = {}

    for block, filename in RAW_BLOCK_FILES.items():
        path = feature_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Missing raw feature CSV for {block}: {path}")

        raw_df = pd.read_csv(path, index_col=0)
        n = min(n_components, raw_df.shape[0], raw_df.shape[1])
        if n < 1:
            raise ValueError(f"Not enough rows/features to fit PCA for {block}: {raw_df.shape}")

        pca = PCA(n_components=n)
        reduced = pca.fit_transform(raw_df.values)
        pcas[block] = pca

        joblib.dump(pca, model_dir / PCA_FILES[block])
        pd.DataFrame(reduced).to_csv(feature_dir / REDUCED_OUTPUT_FILES[block])

        print(f"Saved {block:>8} PCA | raw={raw_df.shape} -> reduced={reduced.shape}")

    # Save expected column order if X_features_401.csv exists
    x_path = feature_dir / "X_features_401.csv"
    if x_path.exists():
        X_ref = pd.read_csv(x_path)
        joblib.dump(X_ref.columns.tolist(), model_dir / "feature_columns.joblib")
        print(f"Saved feature_columns.joblib with {X_ref.shape[1]} columns")
    else:
        print("No X_features_401.csv found. The app can fall back to model.feature_names_in_ if available.")

    return pcas

# Run after the feature pipeline produces the raw feature CSVs:
# pcas = fit_and_save_pca_artifacts(FEATURE_DIR, MODEL_DIR, n_components=50)

## 4. Single-song raw feature extraction

This mirrors the shaping logic from your MTG feature pipeline for a single song.

In [3]:
def pad_or_crop_frames(X, n_frames=776):
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError(f"Expected 2D feature matrix, got {X.shape}")
    t = X.shape[1]
    if t == 0:
        raise ValueError("Feature matrix has zero frames.")
    if t >= n_frames:
        return X[:, :n_frames]
    reps = int(math.ceil(n_frames / t))
    return np.tile(X, reps)[:, :n_frames]


def extract_raw_feature_blocks(audio_path, n_frames=776, duration=180, sr_features=2205):
    audio_path = str(audio_path)

    # Tempo: original-sample-rate load, matching your project notebook
    y_tempo, sr_tempo = librosa.load(audio_path, sr=None, mono=True, duration=duration)
    onset_env = librosa.onset.onset_strength(y=y_tempo, sr=sr_tempo)
    try:
        tempo_arr = librosa.feature.rhythm.tempo(onset_envelope=onset_env, sr=sr_tempo)
    except AttributeError:
        tempo_arr = librosa.beat.tempo(onset_envelope=onset_env, sr=sr_tempo)
    tempo = float(tempo_arr[0])

    # Main feature extraction: your notebook used sr=2205 and fixed 776 frames
    y, sr = librosa.load(audio_path, sr=sr_features, mono=True, duration=duration)

    mfccs = librosa.feature.mfcc(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    S = np.abs(librosa.stft(y))
    contrast = librosa.feature.spectral_contrast(S=S, sr=22050)  # kept consistent with notebook
    flat = librosa.feature.spectral_flatness(y=y)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=y, sr=22050)             # kept consistent with notebook

    blocks = {
        "mfcc": pad_or_crop_frames(mfccs, n_frames).reshape(1, -1),
        "chroma": pad_or_crop_frames(chroma, n_frames).reshape(1, -1),
        "cent": pad_or_crop_frames(cent, n_frames).reshape(1, -1),
        "bw": pad_or_crop_frames(bw, n_frames).reshape(1, -1),
        "contrast": pad_or_crop_frames(contrast, n_frames).reshape(1, -1),
        "flat": pad_or_crop_frames(flat, n_frames).reshape(1, -1),
        "rolloff": pad_or_crop_frames(rolloff, n_frames).reshape(1, -1),
        "tonnetz": pad_or_crop_frames(tonnetz, n_frames).reshape(1, -1),
    }
    return blocks, {"tempo": tempo}

## 5. Transform one song into the model-ready feature row

In [4]:
BLOCK_COLUMN_NAMES = {
    "bw": "spectral bandwidth",
    "cent": "spectral centroid",
    "chroma": "chromagram",
    "contrast": "spectral contrast",
    "flat": "spectral flatness",
    "mfcc": "MFCCs",
    "rolloff": "spectral rolloff",
    "tonnetz": "tonnetz",
}

# Same final order you used when creating X_features_401.csv
MODEL_BLOCK_ORDER = ["bw", "cent", "chroma", "contrast", "flat", "mfcc", "rolloff", "tonnetz"]


def load_pca_artifacts(model_dir=MODEL_DIR):
    model_dir = Path(model_dir)
    pcas = {}
    for block, filename in PCA_FILES.items():
        path = model_dir / filename
        if not path.exists():
            raise FileNotFoundError(
                f"Missing PCA artifact: {path}/n"

                "Run fit_and_save_pca_artifacts(...) or copy the PCA artifact into models/."
            )
        pcas[block] = joblib.load(path)
    return pcas


def make_block_df(values, base_name):
    return pd.DataFrame(values, columns=[f"{base_name} {i+1}" for i in range(values.shape[1])])


def load_expected_columns(depth_model=None, model_dir=MODEL_DIR):
    col_path = Path(model_dir) / "feature_columns.joblib"
    if col_path.exists():
        return joblib.load(col_path)
    if depth_model is not None and hasattr(depth_model, "feature_names_in_"):
        return list(depth_model.feature_names_in_)
    return None


def audio_to_feature_row(audio_path, pcas=None, expected_columns=None):
    if pcas is None:
        pcas = load_pca_artifacts(MODEL_DIR)

    raw_blocks, scalar = extract_raw_feature_blocks(audio_path)

    frames = []
    for block in MODEL_BLOCK_ORDER:
        reduced = pcas[block].transform(raw_blocks[block])
        frames.append(make_block_df(reduced, BLOCK_COLUMN_NAMES[block]))

    X_song = pd.concat(frames + [pd.DataFrame({"tempo": [scalar["tempo"]]})], axis=1)

    if expected_columns is not None:
        missing = [c for c in expected_columns if c not in X_song.columns]
        if missing:
            raise ValueError(f"Missing expected columns: {missing[:10]} ... total missing={len(missing)}")
        X_song = X_song.reindex(columns=expected_columns)

    return X_song

## 6. Load model and predict one file

In [5]:
def load_depth_system(model_path=MODEL_PATH, model_dir=MODEL_DIR):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model file: {model_path}")
    model = joblib.load(model_path)
    pcas = load_pca_artifacts(model_dir)
    expected_cols = load_expected_columns(model, model_dir)
    print("Expected columns:", len(expected_cols) if expected_cols is not None else "not found")
    return model, pcas, expected_cols


def predict_depth_from_audio(audio_path, model=None, pcas=None, expected_cols=None):
    if model is None or pcas is None:
        model, pcas, expected_cols = load_depth_system()
    X_song = audio_to_feature_row(audio_path, pcas=pcas, expected_columns=expected_cols)
    pred_scaled = float(np.asarray(model.predict(X_song)).ravel()[0])
    pred_scaled_clipped = float(np.clip(pred_scaled, 0, 1))
    pred_likert = 4 * pred_scaled_clipped - 2
    return {
        "scaled_depth": pred_scaled_clipped,
        "raw_model_output": pred_scaled,
        "likert_estimate_minus2_to_2": pred_likert,
        "n_features": int(X_song.shape[1]),
    }

# Example:
# model, pcas, expected_cols = load_depth_system()
# predict_depth_from_audio("path/to/song.mp3", model, pcas, expected_cols)

## 7. Gradio interface

Run this cell to open the upload app. In Colab, change `interface.launch()` to `interface.launch(share=True)`.

In [7]:
model, pcas, expected_cols = load_depth_system(MODEL_PATH, MODEL_DIR)

def gradio_predict(audio_file):
    if audio_file is None:
        return "Upload an audio file first.", None, None
    try:
        result = predict_depth_from_audio(audio_file, model=model, pcas=pcas, expected_cols=expected_cols)
        scaled = result["scaled_depth"]
        likert = result["likert_estimate_minus2_to_2"]
        raw = result["raw_model_output"]
        summary = (
            f"Predicted scaled depth: {scaled:.3f}/n"
            f"Estimated original Likert depth (-2 to 2): {likert:.3f}/n"
            f"Raw model output: {raw:.3f}/n"
            f"Feature count: {result['n_features']}"
        )
        return summary, scaled, likert
    except Exception as e:
        return f"Prediction failed: {e}", None, None

interface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Audio(type="filepath", label="Drop in a song file"),
    outputs=[
        gr.Textbox(label="Prediction summary", lines=5),
        gr.Number(label="Scaled depth score (0–1)"),
        gr.Number(label="Likert estimate (-2 to 2)"),
    ],
    title="Music Depth Predictor",
    description="Upload an audio file. The app extracts handcrafted features, applies saved PCA transforms, and predicts musical depth.",
)

interface.launch()

Expected columns: 401
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 8. Optional: write a standalone `app.py`

This cell exports a standalone Gradio app. Place `app.py` beside a `models/` folder containing `depth_model.joblib`, `feature_columns.joblib`, and the PCA artifacts.

In [ ]:
APP_CODE = '\nfrom pathlib import Path\nimport math\nimport warnings\n\nimport joblib\nimport librosa\nimport numpy as np\nimport pandas as pd\nimport gradio as gr\n\nwarnings.filterwarnings("ignore", category=UserWarning)\n\nPROJECT_ROOT = Path(__file__).resolve().parent\nMODEL_DIR = PROJECT_ROOT / "models"\nMODEL_PATH = MODEL_DIR / "depth_model.joblib"\n\nPCA_FILES = {\n    "mfcc": "pca_mfcc.joblib",\n    "chroma": "pca_chroma.joblib",\n    "cent": "pca_cent.joblib",\n    "bw": "pca_bw.joblib",\n    "contrast": "pca_contrast.joblib",\n    "flat": "pca_flat.joblib",\n    "rolloff": "pca_rolloff.joblib",\n    "tonnetz": "pca_tonnetz.joblib",\n}\n\nBLOCK_COLUMN_NAMES = {\n    "bw": "spectral bandwidth",\n    "cent": "spectral centroid",\n    "chroma": "chromagram",\n    "contrast": "spectral contrast",\n    "flat": "spectral flatness",\n    "mfcc": "MFCCs",\n    "rolloff": "spectral rolloff",\n    "tonnetz": "tonnetz",\n}\nMODEL_BLOCK_ORDER = ["bw", "cent", "chroma", "contrast", "flat", "mfcc", "rolloff", "tonnetz"]\n\ndef pad_or_crop_frames(X, n_frames=776):\n    X = np.asarray(X)\n    t = X.shape[1]\n    if t >= n_frames:\n        return X[:, :n_frames]\n    reps = int(math.ceil(n_frames / t))\n    return np.tile(X, reps)[:, :n_frames]\n\ndef extract_raw_feature_blocks(audio_path, n_frames=776, duration=180, sr_features=2205):\n    y_tempo, sr_tempo = librosa.load(str(audio_path), sr=None, mono=True, duration=duration)\n    onset_env = librosa.onset.onset_strength(y=y_tempo, sr=sr_tempo)\n    try:\n        tempo_arr = librosa.feature.rhythm.tempo(onset_envelope=onset_env, sr=sr_tempo)\n    except AttributeError:\n        tempo_arr = librosa.beat.tempo(onset_envelope=onset_env, sr=sr_tempo)\n    tempo = float(tempo_arr[0])\n\n    y, sr = librosa.load(str(audio_path), sr=sr_features, mono=True, duration=duration)\n    mfccs = librosa.feature.mfcc(y=y, sr=sr)\n    chroma = librosa.feature.chroma_stft(y=y, sr=sr)\n    cent = librosa.feature.spectral_centroid(y=y, sr=sr)\n    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)\n    S = np.abs(librosa.stft(y))\n    contrast = librosa.feature.spectral_contrast(S=S, sr=22050)\n    flat = librosa.feature.spectral_flatness(y=y)\n    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)\n    tonnetz = librosa.feature.tonnetz(y=y, sr=22050)\n    return {\n        "mfcc": pad_or_crop_frames(mfccs, n_frames).reshape(1, -1),\n        "chroma": pad_or_crop_frames(chroma, n_frames).reshape(1, -1),\n        "cent": pad_or_crop_frames(cent, n_frames).reshape(1, -1),\n        "bw": pad_or_crop_frames(bw, n_frames).reshape(1, -1),\n        "contrast": pad_or_crop_frames(contrast, n_frames).reshape(1, -1),\n        "flat": pad_or_crop_frames(flat, n_frames).reshape(1, -1),\n        "rolloff": pad_or_crop_frames(rolloff, n_frames).reshape(1, -1),\n        "tonnetz": pad_or_crop_frames(tonnetz, n_frames).reshape(1, -1),\n    }, {"tempo": tempo}\n\ndef load_pca_artifacts():\n    return {block: joblib.load(MODEL_DIR / fname) for block, fname in PCA_FILES.items()}\n\ndef make_block_df(values, base_name):\n    return pd.DataFrame(values, columns=[f"{base_name} {i+1}" for i in range(values.shape[1])])\n\ndef load_expected_columns(model):\n    path = MODEL_DIR / "feature_columns.joblib"\n    if path.exists():\n        return joblib.load(path)\n    if hasattr(model, "feature_names_in_"):\n        return list(model.feature_names_in_)\n    return None\n\ndef audio_to_feature_row(audio_path, pcas, expected_columns):\n    raw_blocks, scalar = extract_raw_feature_blocks(audio_path)\n    frames = []\n    for block in MODEL_BLOCK_ORDER:\n        reduced = pcas[block].transform(raw_blocks[block])\n        frames.append(make_block_df(reduced, BLOCK_COLUMN_NAMES[block]))\n    X_song = pd.concat(frames + [pd.DataFrame({"tempo": [scalar["tempo"]]})], axis=1)\n    if expected_columns is not None:\n        X_song = X_song.reindex(columns=expected_columns)\n        if X_song.isna().any().any():\n            raise ValueError("Feature-column mismatch after reindexing. Check feature_columns.joblib and PCA artifacts.")\n    return X_song\n\nmodel = joblib.load(MODEL_PATH)\npcas = load_pca_artifacts()\nexpected_cols = load_expected_columns(model)\n\ndef predict(audio_file):\n    if audio_file is None:\n        return "Upload an audio file first.", None, None\n    X = audio_to_feature_row(audio_file, pcas, expected_cols)\n    raw_pred = float(np.asarray(model.predict(X)).ravel()[0])\n    scaled = float(np.clip(raw_pred, 0, 1))\n    likert = 4 * scaled - 2\n    return (\n        f"Predicted scaled depth: {scaled:.3f}\\n"\n        f"Estimated original Likert depth (-2 to 2): {likert:.3f}\\n"\n        f"Raw model output: {raw_pred:.3f}\\n"\n        f"Feature count: {X.shape[1]}",\n        scaled,\n        likert,\n    )\n\napp = gr.Interface(\n    fn=predict,\n    inputs=gr.Audio(type="filepath", label="Drop in a song file"),\n    outputs=[gr.Textbox(label="Prediction summary", lines=5), gr.Number(label="Scaled depth"), gr.Number(label="Likert estimate")],\n    title="Music Depth Predictor",\n    description="Handcrafted audio features + saved PCA transforms + saved depth model.",\n)\n\nif __name__ == "__main__":\n    app.launch()\n'
with open("app.py", "w", encoding="utf-8") as f:
    f.write(APP_CODE)
print("Wrote app.py")